# Runoff Sensitivity  to Mass Balance Parameters

Goals of this notebook: To explore the sensitivity of the runoff and OGGM's hydrological components to the mass balance parameters.

Our other hydrology tutorials introduce the idea of hydrology in the OGGM. It is recommended to consult the previous tutorials to gain an understanding first as to how hydrology works in the OGGM. 

[Glaciers as water resources: Part 1](https://edu-notebooks.oggm.org/oggm-edu/glacier_water_resources.html)

[Glaciers as water resources: Part 2](https://edu-notebooks.oggm.org/oggm-edu/glacier_water_resources_projections.html)

## Set Up

First install required packages to run this tutorial and initialize our glacier directories!

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib

from oggm import cfg, utils, workflow, tasks, DEFAULT_BASE_URL
from oggm.core import massbalance
from oggm.core.massbalance import MultipleFlowlineMassBalance

In [ ]:
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='OGGM-runoff', reset=True)
cfg.PARAMS['store_model_geometry'] = True

We start from a well known glacier in the Austrian Alps, Hintereisferner. But you can choose any other glacier, e.g. from [this list](https://github.com/OGGM/oggm-sample-data/blob/master/wgms/rgi_wgms_links_20220112.csv)

In [ ]:
# Hintereisferner
rgi_id = 'RGI60-11.00897'

# We pick the elevation-bands glaciers because they run a bit faster - but they create more step changes in the area outputs
gdir_hef = workflow.init_glacier_directories([rgi_id], from_prepro_level=4, prepro_border=160, prepro_base_url=DEFAULT_BASE_URL)[0]

# An Introduction to Sensitivity Analysis

**Sensitivity Analysis** investigates how the variation in the output of a numerical model can be attributed to variations of its input factors [1](https://www.sciencedirect.com/science/article/pii/S1364815216300287).

In this tutorial, we perform a simple, exploratory one-at-a-time sensitivity analysis to investigate the effects of the mass balance parameters on the glaciohydrological model outputs. We will focus on the mass balance parameters: the melt factor, temperature bias and precipitation factor.

This will allow us to understand the relationship between our parameters and model output, for example if one parameter is changed, how much does this affect our model output?

# Simple Sensitivity Analysis Experiment

### Temperature Bias

We will begin by investigating the sensitvity of the runoff to one parameter at a time, we will start with **temperature bias**. Below, we will vary only the temperature bias, and fix the melt factor and the precipitation factor.

In [ ]:
temp_bias_dict = {}
file_id = '_sens'

for temp_bias in np.arange(-5,5.0,0.5): # We are varying the temperature bias

	# For each temperature bias, we create a mass balance model with the same melt factor and precipitation factor, but with the new temperature bias
	mb_model = MultipleFlowlineMassBalance(
    	gdir_hef,
    	mb_model_class=massbalance.MonthlyTIModel,
    	temp_bias=float(temp_bias), # Vary the temperature bias
		melt_f=5.0, # Fix melt factor to 5.0 for all runs, to see the effect of the temperature bias alone
		prcp_fac=2.5, # Fix precipitation factor to 2.5 for all runs, to see the effect of the temperature bias alone
    	check_calib_params=False,
    	) 
	
	# We are using the task run_with_hydro to store hydrological outputs along with the usual glaciological outputs
	# Run this with the our 2 fixed mass balance parameters and our varying temperature bias
	tasks.run_with_hydro(gdir_hef,  # Run on the selected glacier
				   		run_task=tasks.run_from_climate_data, # Run from climate data
				   		mb_model=mb_model, # use the mass balance model with the new temp_bias
						ys=2000,  # Period which we will average and constantly repeat
						init_model_yr=2000, # Start from spinup year 2000
						init_model_filesuffix='_spinup_historical',  # use the previous run as initial state
						store_monthly_hydro=True,  # Monthly outputs provide additional information
						output_filesuffix=file_id)  # an identifier for the output file, to read it later
	
	# Now read the hydrological outputs for this run
	with xr.open_dataset(gdir_hef.get_filepath('model_diagnostics', filesuffix=file_id)) as ds_sens:
		# The last step of hydrological output is NaN (we can't compute it for this year)
		ds_sens = ds_sens.isel(time=slice(0, -1)).load()

	sel_vars = [v for v in ds_sens.variables if 'month_2d' not in ds_sens[v].dims]
	df_annual_sens = ds_sens[sel_vars].to_dataframe()

	# Calculating the runoff for the temperature bias value and storing it in a dictionary to plot
	temp_bias_dict[temp_bias] = (df_annual_sens['melt_off_glacier']+df_annual_sens['melt_on_glacier']+df_annual_sens['liq_prcp_off_glacier']+ df_annual_sens['liq_prcp_on_glacier'])*1e-9 # Convert from Kilograms to Megatonnes per year (Mt/yr) for easier plotting

Now, for each temperature bias value, let's plot the mean runoff against the temperature bias, and plot the runoff time series to undestand how this varies across our range of temperature bias values.

In [ ]:
# Now let's get a nice colormap centered at temp_bias=0
norm = matplotlib.colors.Normalize(vmin=-5, vmax=5.01)
colors_temp_bias = plt.get_cmap('coolwarm')

fig, axs = plt.subplots(1,2,figsize=(14,6))
for j, temp_bias in enumerate(temp_bias_dict.keys()):
    axs[0].plot(temp_bias, temp_bias_dict[temp_bias].mean(), 'o',
             color=colors_temp_bias(norm(temp_bias))) 
axs[0].set_ylabel('Mean Runoff (Mt/yr)')
axs[0].set_xlabel('temp_bias (°C)');
axs[0].set_title('Mean Runoff vs Temperature Bias')


for temp_bias in temp_bias_dict.keys():
    axs[1].plot(np.arange(2000,2020,1),
              temp_bias_dict[temp_bias].values, '-', 
             color=colors_temp_bias(norm(temp_bias)),
             label=temp_bias)
axs[1].set_ylabel('Runoff (Mt/yr)')
axs[1].set_xlabel('Year')
axs[1].legend(title='temp_bias:', bbox_to_anchor=(1,1))
axs[1].set_xticks(np.arange(2000,2020,2));
axs[1].set_title('Runoff Time Series for Different Temperature Biases')

fig.suptitle("Runoff and Temperature Bias - {}".format(gdir_hef.rgi_id));
plt.tight_layout()

We can see that the runoff appears to be sensitive to the temperature bias! In the graph on the left, we can see the mean runoff increases as we increase the temperature bias, and on the graph on the right, we can see how this affects the runoff annually.

### Precipitation Factor

Now lets explore what happens when we vary the **precipitation factor**, and fix our other two mass balance parameters.

In [ ]:
prcp_fac_dict = {}

for prcp_fac in np.arange(0.1,10,0.5): # Now we are varying the precipitation factor
	
	# For each precipitation factor, we create a mass balance model with the same melt factor and temperature bias, but with the new precipitation factor
	mb_model = MultipleFlowlineMassBalance(
    	gdir_hef,
    	mb_model_class=massbalance.MonthlyTIModel,
    	prcp_fac=float(prcp_fac),
		melt_f=5.0, # Fix melt factor
		temp_bias=0, # Fix the temperature bias to 0
    	check_calib_params=False,
    	) 
	
	# We are using the task run_with_hydro to store hydrological outputs along with the usual glaciological outputs
	# Run this with the our 2 fixed mass balance parameters and our varying precipitation factor
	tasks.run_with_hydro(gdir_hef,  # Run on the selected glacier
				   		run_task=tasks.run_from_climate_data,
				   		mb_model=mb_model, # use the mass balance model with the new prcp_fac
						ys=2000,  # Period which we will average and constantly repeat
						init_model_yr=2000, # Start from spinup year 2000
						init_model_filesuffix='_spinup_historical',  # use the previous run as initial state
						store_monthly_hydro=True,  # Monthly outputs provide additional information
						output_filesuffix=file_id)  # an identifier for the output file, to read it later
	
	# Reading the glaciological and hydrological outputs for this run again
	with xr.open_dataset(gdir_hef.get_filepath('model_diagnostics', filesuffix=file_id)) as ds_sens:
		# The last step of hydrological output is NaN (we can't compute it for this year)
		ds_sens = ds_sens.isel(time=slice(0, -1)).load()
	sel_vars = [v for v in ds_sens.variables if 'month_2d' not in ds_sens[v].dims]
	df_annual_sens = ds_sens[sel_vars].to_dataframe()

	# Calculating the runoff
	prcp_fac_dict[prcp_fac] = (df_annual_sens['melt_off_glacier']+df_annual_sens['melt_on_glacier']+df_annual_sens['liq_prcp_off_glacier']+ df_annual_sens['liq_prcp_on_glacier']) * 1e-9

Now let's plot to see our results.

In [ ]:
# Now we can centre the colormap around prcp_fac
norm = matplotlib.colors.Normalize(vmin=0.1, vmax=10)
colors_prcp_fac = plt.get_cmap('coolwarm')

fig, axs = plt.subplots(1,2,figsize=(14,6))
for j, prcp_fac in enumerate(prcp_fac_dict.keys()):
    axs[0].plot(prcp_fac, prcp_fac_dict[prcp_fac].mean(), 'o',
             color=colors_prcp_fac(norm(prcp_fac))) 
axs[0].set_ylabel('Mean Runoff (Mt/yr)')
axs[0].set_xlabel('Precipitation factor');
axs[0].set_title('Mean Runoff vs Precipitation Factor')

for prcp_fac in prcp_fac_dict.keys():
    axs[1].plot(np.arange(2000,2020,1),
              prcp_fac_dict[prcp_fac].values, '-', 
             color=colors_prcp_fac(norm(prcp_fac)),
             label=prcp_fac)
axs[1].set_ylabel('Runoff (Mt/yr)')
axs[1].set_xlabel('Year')
axs[1].legend(title='Precipitation factor:', bbox_to_anchor=(1,1))
axs[1].set_xticks(np.arange(2000,2020,2));
axs[1].set_title('Runoff Time Series for Different Precipitation Factors')

fig.suptitle("Runoff and Precipitation Factor - {}".format(gdir_hef.rgi_id));
plt.tight_layout()

We can see that again, the runoff appears sensitive to the precipitation factor! And that as the precipitation factor increases, as does the runoff.

### Melt Factor

Finally, let's see what happens when we alter the melt factor and fix the remaining 2 mass balance parameters!

In [ ]:
melt_f_dict = {}

for melt_f in np.arange(1.5,17,1.0):  # Now we are varying the melt factor
	
	# For each melt factor, we create a mass balance model with the same precipitation factor and temperature bias, but now with the new melt factor
	mb_model = MultipleFlowlineMassBalance(
    	gdir_hef,
    	mb_model_class=massbalance.MonthlyTIModel,
    	melt_f=float(melt_f),
		prcp_fac=2.5,
		temp_bias=0,
    	check_calib_params=False,
    	) 
	
	# Run this with the our 2 fixed mass balance parameters and our varying melt factor
	tasks.run_with_hydro(gdir_hef,  # Run on the selected glacier
						run_task=tasks.run_from_climate_data, # running from observed climate data
						ys=2000,  # Period which we will average and constantly repeat
						init_model_yr=2000, # Start from spinup year 2000
						init_model_filesuffix='_spinup_historical',  # use the previous run as initial state
						mb_model=mb_model, # use the mass balance model with the new melt_f
						store_monthly_hydro=True,  # Monthly outputs provide additional information
						output_filesuffix=file_id)  # an identifier for the output file, to read it later
	
	# Reading the glaciological and hydrological outputs
	with xr.open_dataset(gdir_hef.get_filepath('model_diagnostics', filesuffix=file_id)) as ds_sens:
		# The last step of hydrological output is NaN (we can't compute it for this year)
		ds_sens = ds_sens.isel(time=slice(0, -1)).load()

	sel_vars = [v for v in ds_sens.variables if 'month_2d' not in ds_sens[v].dims]
	df_annual_sens = ds_sens[sel_vars].to_dataframe()

	# Calculating the runoff
	melt_f_dict[melt_f] = (df_annual_sens['melt_off_glacier']+df_annual_sens['melt_on_glacier']+df_annual_sens['liq_prcp_off_glacier']+ df_annual_sens['liq_prcp_on_glacier']) * 1e-9

Now we plot:

In [ ]:
# let's get a nice colormap centered around melt_f
norm = matplotlib.colors.Normalize(vmin=1.5, vmax=17)
colors_melt_f = plt.get_cmap('coolwarm')

fig, axs = plt.subplots(1,2,figsize=(14,6))
for j, melt_f in enumerate(melt_f_dict.keys()):
    axs[0].plot(melt_f, melt_f_dict[melt_f].mean(), 'o',
             color=colors_melt_f(norm(melt_f))) 
axs[0].set_ylabel('Runoff (Mt/yr)')
axs[0].set_xlabel('Melt factor');
axs[0].set_title('Mean Runoff vs Melt Factor')


for melt_f in melt_f_dict.keys():
    axs[1].plot(np.arange(2000,2020,1),
              melt_f_dict[melt_f].values, '-', 
             color=colors_melt_f(norm(melt_f)),
             label=melt_f)
axs[1].set_ylabel('Runoff (Mt/yr)')
axs[1].set_xlabel('Year')
axs[1].legend(title='Melt factor:', bbox_to_anchor=(1,1))
axs[1].set_xticks(np.arange(2000,2020,2));
axs[1].set_title('Runoff Time Series for Different Melt Factors')

fig.suptitle("Runoff and Melt Factor - {}".format(gdir_hef.rgi_id));
plt.tight_layout()

Again, we can see that the runoff is sensitive to the melt factor too! And that, as the melt factor increases, so does the runoff.

We have explored the relationship between the mass balance parameters and the runoff, but what about other glaciohydrological outputs?

# Exploring other Glaciohydrological outputs in the OGGM

**First let's start with a definition!** In this next section we will be investigating the melt contribution to runoff, defined as: $ \frac{\text{melt on glacier}}{\text{runoff}}$. 

This represents how much of the total runoff can be attributed to the glacial melt (i.e. the melt water produced from the currently glaciated area).

Below we will investigate both the melt contribution, and `melt_on_glacier` sensitivity to the precipitation factor. 

We run a final set of simulations to obtain the glaciohydrological outputs from the OGGM for our sensitivity study.

In [ ]:
# Varying prcp_fac between a range of values with a step of 1.0
pd_prcp_sens = pd.DataFrame(index=np.arange(0.1, 10.0, 0.5))
file_id = '_sens'

for pf in pd_prcp_sens.index:

    mb_model = MultipleFlowlineMassBalance(
        gdir_hef,
        mb_model_class=massbalance.MonthlyTIModel,
        prcp_fac=float(pf),
        melt_f=5.0,
        temp_bias=0,
        check_calib_params=False,
        ) 

    # We are using the task run_with_hydro to store hydrological outputs along with the usual glaciological outputs
    # Run this again with the calibrated parameters
    tasks.run_with_hydro(gdir_hef,  # Run on the selected glacier
                        run_task=tasks.run_from_climate_data, # running from observed climate data
                        ys=2000,  # Period which we will average and constantly repeat
                        init_model_yr=2000, # Start from spinup year 2000
                        init_model_filesuffix='_spinup_historical',  # use the previous run as initial state
                        mb_model=mb_model, # use the mass balance model with the new melt_f
                        store_monthly_hydro=True,  # Monthly outputs provide additional information
                        output_filesuffix=file_id);  # an identifier for the output file, to read it later

    with xr.open_dataset(gdir_hef.get_filepath('model_diagnostics', filesuffix=file_id)) as ds_sens:
        # The last step of hydrological output is NaN (we can't compute it for this year)
        ds_sens = ds_sens.isel(time=slice(0, -1)).load()

    # Plot the runoff again for the calibrated melt_f parameter
    sel_vars = [v for v in ds_sens.variables if 'month_2d' not in ds_sens[v].dims]
    df_annual_sens = ds_sens[sel_vars].to_dataframe()

    pd_prcp_sens.loc[pf, 'melt_off_glacier'] = df_annual_sens['melt_off_glacier'].mean() * 1e-9
    pd_prcp_sens.loc[pf, 'melt_on_glacier'] = df_annual_sens['melt_on_glacier'].mean() * 1e-9
    pd_prcp_sens.loc[pf, 'liq_prcp_off_glacier'] = df_annual_sens['liq_prcp_off_glacier'].mean() * 1e-9
    pd_prcp_sens.loc[pf, 'liq_prcp_on_glacier'] = df_annual_sens['liq_prcp_on_glacier'].mean() * 1e-9
    pd_prcp_sens.loc[pf, 'runoff'] = pd_prcp_sens.loc[pf, 'melt_off_glacier'] + pd_prcp_sens.loc[pf, 'melt_on_glacier'] + pd_prcp_sens.loc[pf, 'liq_prcp_off_glacier'] + pd_prcp_sens.loc[pf, 'liq_prcp_on_glacier']

Now plotting to investigate the sensitivity of the `melt_on_glacier` and the melt contribution to the precipitation factor.

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(14,6))

for pf in pd_prcp_sens.index:
    axs[0].scatter(
        pf,
        pd_prcp_sens.loc[pf, 'melt_on_glacier'], # melt on glacier
        color='blue'
    )
    
    axs[1].scatter(
        pf,
        pd_prcp_sens.loc[pf, 'melt_on_glacier']/pd_prcp_sens.loc[pf, 'runoff'], # melt contribution
        color='blue'
    )


axs[0].set_xlabel('Precipitation factor')
axs[0].set_ylabel('Mean annual melt on glacier (Mt/yr)')
axs[0].set_title('Sensitivity of Melt on Glacier to Precipitation Factor')

axs[1].set_xlabel('Precipitation factor')
axs[1].set_ylabel('Mean Glacial Melt Contribution')
axs[1].set_title('Sensitivity of Glacial Melt Contribution to the Precipitation Factor')

plt.show()


The above graph shows that these glaciohydrological outputs are both sensitive to the precipitation factor changing!

We can see that as the precipitation factor increases, the mean annual melt on glacier increases. 

However, when we divide this value by the total runoff to derive the melt contribution, this results in a decrease in the melt contribution as the precipitation factor increases.

Let's think about why there might be an increase in the melt on glacier with an increasing precipitation factor, but a decreasing glacial melt contribution? We can start by investigating how much the total runoff increases, compared to the melt on glacier, and this might help us answer the question!


In [ ]:
fig, axs = plt.subplots()

for i, pf in enumerate(pd_prcp_sens.index):
    axs.scatter(
        pf,
        pd_prcp_sens.loc[pf, 'melt_on_glacier'],
        color='blue',
        label='Melt on glacier' if i == 0 else None
    )
    axs.scatter(
        pf,
        pd_prcp_sens.loc[pf, 'runoff'],
        color='red',
        label='Runoff' if i == 0 else None
    )
    
axs.set_xlabel('Precipitation factor')
axs.set_ylabel('Glaciohydrological components')
axs.set_title('Sensitivity of Melt on Glacier to Precipitation Factor')

plt.legend()

plt.show()


We can see that the total runoff is more sensitive to the changes in the precipitation factor and is increasing at a much more rapid rate when we increase the precipitation factor. Therefore this causes a decrease in the melt contribution makes sense.

This shows us that different outputs can have different sensitivities to the same changing input parameters, and can lead to some interesting results!

# Tutorial take-aways
- Sensitivity analysis is a useful tool to understand the relationship between our model inputs and outputs
- One-at-a-time sensitivity analysis is an exploratory tool to help us better investigate model behaviour.
- Different outputs can exhibit different sensitivities for the same range of input values.
- The model sensitivity depends greatly on the chosen inputs (what they are, and their ranges) and the outputs we are considering.


This is a very simple application of sensitivity analysis applied to the OGGM, but these insights can motivate further use of sensitivity analysis on a larger-scale. There are future plans to integrate the [Sensitivity Analysis For Everyone Toolbox (SAFE)](https://safetoolbox.github.io/) with OGGM for more complex sensitivity analysis applications!


# References

1. Francesca Pianosi, Keith Beven, Jim Freer, Jim W. Hall, Jonathan Rougier, David B. Stephenson, Thorsten Wagener,
Sensitivity analysis of environmental models: A systematic review with practical workflow, Environmental Modelling & Software, Volume 79, 2016, Pages 214-232, ISSN 1364-8152, https://doi.org/10.1016/j.envsoft.2016.02.008.